# Lab 4: Markov Chain and Q-Learning

## Markov Chain
* **Shift from one state to another**
* **Depend on next step not sequence of past step.**
* **Memoryless**

### Properties
1. **State space**
2. **Transition Probability Matrix**
3. **Stationary distribution**
   * No change probability distribution.

## Markov Decision Process (MDP)
>> Sequential Decision making problem where outcome of an action is uncertain.

>> Components: State (S), Action (A), transition (P), Reward function (R), $\gamma$ (discount Rate)

## Q-Learning
* Trial and Error algorithm
* Learn the long term value
* Specific action in specific situation
* **Q** $\rightarrow$ Quality
* Lookup table (Q-table)
* *If state "S" and take action "A", How good will my total future pay off?*

### Mathematical Equation
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Where:
* $Q(s,a)$ (old value) $\rightarrow$ agent current think action a is worth
* $\alpha$ (learning rate) $\rightarrow$ "0" and "1" [0: nothing learn, 1: Learn new]
* $R$ (immediate Reward) $\rightarrow$ penalty score
* $\gamma$ (Discount factor) $\rightarrow$ "0" and "1" [reward against immediate one]
* $s'$ $\rightarrow$ Next state
* $\max_{a'} Q(s', a')$ $\rightarrow$ Best Expected future value

## Example: 4x4 Grid World

**Grid:**
| S0 | S1 | S2 | S3 |
|---|---|---|---|
| S4 | **S5** | S6 | S7 |
| S8 | S9 | **S10**| S11|
| S12| S13| S14| **S15**|

* **Start State:** S0
* **Goal State:** S15
* **Obstacles:** S5 & S10
* **Parameters:** $\alpha = 0.5$, $\gamma = 0.9$
* **Rewards:** 
  * $R = +10$ (Goal)
  * $R = -10$ (obstacle)
  * $R = -1$ (Normal)
* **Actions:** $\uparrow$ (U), $\downarrow$ (D), $\leftarrow$ (L), $\rightarrow$ (R)

### Initial Step:
**Q-table:** Intialized with zeros for all states ($S$) and actions.

### Episode Walkthrough

**Ep 1:**
* $S0 \xrightarrow{R} S1$
  * $Q(S0, R) = 0 + 0.5[-1 + 0.9(0) - 0] = -0.5$
* $S1 \xrightarrow{D} S5$ *(obstacle)*
  * $Q(S1, D) = 0 + 0.5[-10 + 0.9(0) - 0] = -5$

**Ep 2:**
* $S0 \xrightarrow{D} S4$
  * $Q(S0, D) = -0.5$
* ... *(skipping intermediate steps)* ...
* $S14 \xrightarrow{R} S15$ *(goal)*
  * $Q(S14, R) = 0 + 0.5[10 + 0.9(0) - 0] = 5$

**Ep 3 (Back propagate):**
* $Q(S14, R) = 5$
* $S13 \xrightarrow{R} S14$
  * Assume $Q(S13, R)$ was previously $-0.5$
  * $Q(S13, R) \leftarrow -0.5 + 0.5[-1 + 0.9(5) - (-0.5)]$
  * $= -0.5 + 0.5[-1 + 4.5 + 0.5] = -0.5 + 0.5[4] = -0.5 + 2 = 1.5$

## Code Implementation
Below is the Python code simulating the Q-Learning process on this 4x4 Grid Environment.

In [1]:
import numpy as np
import pandas as pd
import random

# ----------------------------------------------------
# ENVIRONMENT SETUP
# ----------------------------------------------------
grid_size = 4
num_states = grid_size * grid_size
num_actions = 4 # 0: Up, 1: Down, 2: Left, 3: Right
actions = ['Up', 'Down', 'Left', 'Right']
state_labels = [f"S{i}" for i in range(num_states)]

start_state = 0
goal_state = 15
obstacles = [5, 10]

R_normal = -1
R_goal = 10
R_obstacle = -10

# Hyperparameters
alpha = 0.5
gamma = 0.9
epsilon = 0.1 # Exploration rate
episodes = 1000 # Total episodes for training

# Initialize Q-table
Q = np.zeros((num_states, num_actions))

def display_q_table(q_table):
    """Helper function to display Q-table nicely using pandas"""
    df = pd.DataFrame(q_table, index=state_labels, columns=actions)
    display(df.round(2))

# ----------------------------------------------------
# INITIAL OUTPUT
# ----------------------------------------------------
print("----------------------------------------------------")
print("INITIAL OUTPUT")
print("----------------------------------------------------\n")
print("Initial Q Table\n")
display_q_table(Q)
print("\n")

def get_next_state(state, action):
    row = state // grid_size
    col = state % grid_size
    
    if action == 0 and row > 0:       # Up
        next_state = state - grid_size
    elif action == 1 and row < grid_size - 1: # Down
        next_state = state + grid_size
    elif action == 2 and col > 0:       # Left
        next_state = state - 1
    elif action == 3 and col < grid_size - 1: # Right
        next_state = state + 1
    else:
        next_state = state # Hit wall, stay in same state
        
    return next_state

# ----------------------------------------------------
# TRAINING LOOP
# ----------------------------------------------------
print("----------------------------------------------------")
print("STEPWISE TRAINING OUTPUT (First 2 Episodes to demonstrate manual trace)")
print("----------------------------------------------------\n")

for episode in range(episodes):
    state = start_state
    done = False
    
    path_taken = [f"S{state}"]
    total_reward = 0
    
    while not done:
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, 3)
        else:
            action = np.argmax(Q[state, :])
            
        next_state = get_next_state(state, action)
        
        # Check rewards and termination
        if next_state == goal_state:
            reward = R_goal
            done = True
        elif next_state in obstacles:
            reward = R_obstacle
            done = True
        else:
            reward = R_normal
            done = False
            
        total_reward += reward
        path_taken.append(f"S{next_state}")
            
        # Update Q-Value
        max_next_q = np.max(Q[next_state, :]) if not done else 0
        old_q = Q[state, action]
        
        new_q = old_q + alpha * (reward + gamma * max_next_q - old_q)
        Q[state, action] = new_q
        
        # Print stepwise output for the first 2 episodes only to avoid notebook bloat
        if episode < 2:
            print(f"Episode {episode + 1}\n")
            print(f"Current State : S{state}")
            print(f"Action : {actions[action]}")
            print(f"Next State : S{next_state}\n")
            print(f"Reward = {reward}\n")
            print(f"Old Q = {old_q:.2f}\n")
            print(f"Maximum Future Q-value = {max_next_q:.2f}\n")
            print(f"Bellman Equation : Q(S{state}, {actions[action]}) = {old_q:.2f} + {alpha} * [{reward} + {gamma} * {max_next_q:.2f} - {old_q:.2f}]\n")
            print(f"New Q = {new_q:.2f}\n")
            print("Updated Q Table")
            display_q_table(Q)
            print("-" * 50 + "\n")
            
        state = next_state
        
    # Episode Summary for the first 2 episodes
    if episode < 2:
        print("----------------------------------------------------")
        print("EPISODE SUMMARY")
        print("----------------------------------------------------\n")
        print(f"Episode Number : {episode + 1}")
        print(f"Path taken : {' -> '.join(path_taken)}")
        print(f"Total reward : {total_reward}")
        print(f"Goal reached : {'Yes' if state == goal_state else 'No'}\n")
        print("Current Q-table")
        display_q_table(Q)
        print("=" * 50 + "\n")

# ----------------------------------------------------
# FINAL OUTPUT
# ----------------------------------------------------
print("----------------------------------------------------")
print("FINAL OUTPUT")
print("----------------------------------------------------\n")

print(f"Total Episodes : {episodes}")
print("Training Complete\n")

print("Final Learned Q-table\n")
display_q_table(Q)
print("\n")

# Extracting Optimal Policy and Path
optimal_policy = []
optimal_path = [start_state]
state = start_state

for s in range(num_states):
    if s == goal_state:
        optimal_policy.append("Goal")
    elif s in obstacles:
        optimal_policy.append("Obstacle")
    else:
        best_action = np.argmax(Q[s, :])
        optimal_policy.append(actions[best_action])

print("Optimal Policy:")
policy_df = pd.DataFrame({"State": state_labels, "Best Action": optimal_policy})
display(policy_df)
print("\n")

while state != goal_state and state not in obstacles:
    action = np.argmax(Q[state, :])
    next_state = get_next_state(state, action)
    if next_state == state: # Prevents infinite loop if stuck
        break
    optimal_path.append(next_state)
    state = next_state

print("Optimal Path from Start to Goal:")
print(" -> ".join([f"S{s}" for s in optimal_path]))


----------------------------------------------------
INITIAL OUTPUT
----------------------------------------------------

Initial Q Table



,Up,Down,Left,Right
S0,0.0,0.0,0.0,0.0
S1,0.0,0.0,0.0,0.0
S2,0.0,0.0,0.0,0.0
S3,0.0,0.0,0.0,0.0
S4,0.0,0.0,0.0,0.0
S5,0.0,0.0,0.0,0.0
S6,0.0,0.0,0.0,0.0
S7,0.0,0.0,0.0,0.0
S8,0.0,0.0,0.0,0.0
S9,0.0,0.0,0.0,0.0




----------------------------------------------------
STEPWISE TRAINING OUTPUT (First 2 Episodes to demonstrate manual trace)
----------------------------------------------------

Episode 1

Current State : S0
Action : Up
Next State : S0

Reward = -1

Old Q = 0.00

Maximum Future Q-value = 0.00

Bellman Equation : Q(S0, Up) = 0.00 + 0.5 * [-1 + 0.9 * 0.00 - 0.00]

New Q = -0.50

Updated Q Table


,Up,Down,Left,Right
S0,-0.5,0.0,0.0,0.0
S1,0.0,0.0,0.0,0.0
S2,0.0,0.0,0.0,0.0
S3,0.0,0.0,0.0,0.0
S4,0.0,0.0,0.0,0.0
S5,0.0,0.0,0.0,0.0
S6,0.0,0.0,0.0,0.0
S7,0.0,0.0,0.0,0.0
S8,0.0,0.0,0.0,0.0
S9,0.0,0.0,0.0,0.0


--------------------------------------------------

Episode 1

Current State : S0
Action : Down
Next State : S4

Reward = -1

Old Q = 0.00

Maximum Future Q-value = 0.00

Bellman Equation : Q(S0, Down) = 0.00 + 0.5 * [-1 + 0.9 * 0.00 - 0.00]

New Q = -0.50

Updated Q Table


,Up,Down,Left,Right
S0,-0.5,-0.5,0.0,0.0
S1,0.0,0.0,0.0,0.0
S2,0.0,0.0,0.0,0.0
S3,0.0,0.0,0.0,0.0
S4,0.0,0.0,0.0,0.0
S5,0.0,0.0,0.0,0.0
S6,0.0,0.0,0.0,0.0
S7,0.0,0.0,0.0,0.0
S8,0.0,0.0,0.0,0.0
S9,0.0,0.0,0.0,0.0


... (27 outputs left out)

,Up,Down,Left,Right
S0,0.63,-0.43,0.63,1.81
S1,1.81,-10.00,0.63,3.12
S2,3.12,4.58,1.81,4.58
S3,-1.65,6.20,2.57,-1.43
S4,0.63,-2.73,-3.02,-5.00
S5,0.00,0.00,0.00,0.00
S6,3.12,-10.00,-10.00,6.20
S7,4.58,8.00,4.58,6.20
S8,-2.39,-1.16,-2.58,-2.13
S9,-5.00,-0.66,-1.91,-5.00




Optimal Policy:


,State,Best Action
0,S0,Right
1,S1,Right
2,S2,Down
3,S3,Down
4,S4,Up
5,S5,Obstacle
6,S6,Right
7,S7,Down
8,S8,Down
9,S9,Down




Optimal Path from Start to Goal:
S0 -> S1 -> S2 -> S6 -> S7 -> S11 -> S15


## Conclusion and Discussions

**Conclusion:**
In this lab, we successfully explored the concepts of Markov Chains, Markov Decision Processes (MDPs), and the Q-Learning algorithm. Through a practical 4x4 grid example, we manually traced the trial-and-error approach, examining how an agent updates its Q-table using the Bellman equation. By interacting with the environment, receiving penalties for obstacles (-10) and normal moves (-1), and securing a reward (+10) upon reaching the goal, the agent dynamically learns the optimal path from the start state to the goal.

**Discussions:**
1. **Exploration vs. Exploitation:** The trial-and-error nature of Q-Learning highlights the balance required between exploring new paths and exploiting known high-reward paths. In our programmatic implementation, we employ an $\epsilon$-greedy strategy to handle this.
2. **Impact of Hyperparameters:**
    - **Learning Rate ($\alpha=0.5$):** Determines how much new information overrides old information. A value of 0.5 allows the agent to steadily integrate recent experiences.
    - **Discount Factor ($\gamma=0.9$):** A value close to 1 indicates that the agent cares significantly about long-term rewards, which is critical in a grid maze where the immediate reward is usually negative until the final goal is reached.
3. **Memoryless Property:** As part of the Markov property, the agent evaluates the next action strictly based on the current state (like S13) without needing the history of how it arrived there.
4. **Value Backpropagation:** Our manual trace (Ep 3) clearly demonstrated how the high reward received at the goal state (S15) backpropagates to previous states (S14 $\rightarrow$ S13). This allows the entire Q-table to eventually reflect the true "quality" of actions from any state in the grid.